In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "46d487bb9b699b97c7c5fc5edd866e4d2ad7919b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/46d487bb9b699b97c7c5fc5edd866e4d2ad7919b/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Reproducibility and Inspection

Suppose you tweak a single line in a training script and validation accuracy on
Fashion-MNIST edges up by 0.3 percentage points. Before celebrating, you need to
know how much two runs of the *unchanged* script vary on their own. In
that section, we run softmax regression five times differing only
in their random seed and watch final accuracies span one to four percentage
points simply from different initial weights and batch shuffling, swamping a 0.3
point gain completely.

Debugging a crashing training run presents the mirror-image problem: if the loss
explodes to NaN after hours of training, we want to reproduce the exact sequence
of steps leading up to the crash and then peer inside the running network to
pinpoint which layer emitted the first non-finite value.

This section addresses both halves together. First, we make runs repeatable by
controlling pseudorandom number generators—global seeds, private generator
objects, and multi-worker data loaders—and then tackle the trickier GPU wrinkle:
floating-point non-associativity under parallel atomic reductions. Second, we
inspect live models without modifying their code using *hooks* (and their
framework equivalents) to log activation scales, catch the first NaN layer, and
monitor gradient norms. Both halves reinforce each other: internal diagnostics
are only comparable across two code versions when nothing else changes between
runs. On a first pass, feel free to skim the subsections on generator objects
and data-loader workers, revisiting them once your data pipelines perform random
augmentations.

In [1]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

In [2]:
import random

## Seeds and Random Number Generators

Randomness sneaks into a training pipeline from far more places than most
practitioners realize: weight initialization (that section), dropout
masks at every step, epoch-by-epoch minibatch shuffling, random data
augmentations (crops, flips, color jitter), and parallel data-loader worker
processes. Every random number comes from a *pseudorandom number generator*
(PRNG), a deterministic algorithm whose entire sequence is determined by its
initial *seed*. Seeding every generator the pipeline touches therefore locks
down every random draw.

### Global Seeds

Deep learning frameworks either pull random numbers from a shared global
generator or require passing generator state explicitly as an argument. Below we
demonstrate seeding on a small five-step training loop where weights and
synthetic inputs all come from one seeded source.

`tf.keras.utils.set_random_seed(k)` seeds Python's `random`, NumPy's global RNG, and TensorFlow's global RNG in one call.

In [3]:
def train_once(seed):
    tf.keras.utils.set_random_seed(seed)  # seeds Python, NumPy, and TF
    with tf.device('/CPU:0'):  # see Deterministic Arithmetic below
        net = tf.keras.Sequential([
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dense(1)])
        opt = tf.keras.optimizers.SGD(learning_rate=0.1)
        X, y = tf.random.normal((128, 20)), tf.random.normal((128, 1))
        for _ in range(5):
            with tf.GradientTape() as tape:
                loss = tf.reduce_mean((net(X) - y) ** 2)
            grads = tape.gradient(loss, net.trainable_variables)
            opt.apply_gradients(zip(grads, net.trainable_variables))
        return float(loss), tf.identity(net.layers[0].kernel)

loss_a, w_a = train_once(seed=0)
loss_b, w_b = train_once(seed=0)
loss_c, _ = train_once(seed=1)
print('same seed, identical loss and weights:',
      loss_a == loss_b, bool(tf.reduce_all(w_a == w_b)))
print('different seed:', loss_a, 'vs', loss_c)

same seed, identical loss and weights: True True
different seed: 1.0948041677474976 vs 1.0242893695831299


Both runs with `seed=0` match *bit for bit* (every weight and the final loss are
identical) because identical initial weights, data, and deterministic arithmetic
produce identical gradients (we examine operation order in
that section). Changing the seed to `1` yields a
distinct trajectory and loss.

We placed the TensorFlow run on the CPU. On a GPU, the same seeded code
gives equal losses but, in some runs, weights that differ in their last
bits.

### Generator Objects

Relying on a single global PRNG couples every piece of code that draws from it:
adding a single extra layer or a temporary random print during debugging shifts
the entire downstream stream, silently altering train/validation splits, weight
initializations, and dropout masks all at once. A private *generator object*
isolates a stream behind its own seed so unrelated draws elsewhere can never
perturb it.

`tf.random.Generator.from_seed(42)` creates an isolated PRNG stream unaffected by global draws:

In [4]:
g = tf.random.Generator.from_seed(42)
split = tf.argsort(g.uniform((10,)))  # a random permutation from g's stream
_ = tf.random.normal((1000,))  # unrelated draws from the global generator
split_again = tf.argsort(tf.random.Generator.from_seed(42).uniform((10,)))
print(bool(tf.reduce_all(split == split_again)), split.numpy())

True [4 2 7 3 5 8 9 1 0 6]


### Data Loader Workers

When data loaders prepare minibatches in parallel *worker* processes, random
data augmentation introduces two silent failure modes. If workers are spawned via
OS `fork` as clones of the main process without re-seeding, every worker
inherits the exact same generator state and generates identical "random"
augmentations. Conversely, if each worker initializes from OS entropy, no two
runs will ever match.

`tf.data` runs parallel threads inside one process and takes explicit seeds (`Dataset.shuffle(buffer_size, seed=0)`), avoiding forked-RNG duplication by design:

In [5]:
ds = tf.data.Dataset.range(8).shuffle(8, seed=0).batch(4)
print([b.numpy().tolist() for b in ds])  # epoch 1
print([b.numpy().tolist() for b in ds])  # epoch 2: reshuffled, still seeded
ds = tf.data.Dataset.range(8).shuffle(8, seed=0).batch(4)
print([b.numpy().tolist() for b in ds])  # rebuilt pipeline: epoch 1 again

[[5, 6, 7, 2], [1, 3, 4, 0]]
[[4, 6, 7, 2], [0, 5, 3, 1]]
[[5, 6, 7, 2], [1, 3, 4, 0]]


### Randomness as a Value

Every worker bug above stems from hidden mutable generator state being copied or
shared implicitly behind the scenes. Functional random APIs eliminate hidden
state by treating randomness as an explicit value: every random call takes a key, calling it twice
with the same key returns identical numbers, and independent streams come from
explicitly splitting keys. While you can still reuse a key accidentally, the
reuse is right there in the code. The price is plumbing keys through every
stochastic function. A global seed is simply the implicit shorthand for that
same contract, and private generator objects recover explicit isolation wherever
needed.


### The Spread across Seeds

Now we measure the seed-to-seed variance mentioned at the start of the
section. Below we train softmax regression (that section) on
Fashion-MNIST for three epochs on a GPU across five seeds (0 through 4), seeding
every generator involved and returning each run's final validation accuracy and
weights.

In [6]:
def train_classifier(seed, data):
    tf.keras.utils.set_random_seed(seed)  # initialization and shuffle order
    model = d2l.SoftmaxRegression(num_outputs=10, lr=0.1)
    trainer = d2l.Trainer(max_epochs=3, num_gpus=1)
    trainer.board.display = False
    trainer.fit(model, data)
    return (trainer.history['val_acc'][-1],
            tf.identity(model.net.layers[1].kernel))

data = d2l.FashionMNIST(batch_size=256)
runs = [train_classifier(seed, data) for seed in range(5)]
accs = 100 * np.array([acc for acc, _ in runs])
print('validation accuracy (%):', ' '.join(f'{a:.1f}' for a in accs))
print(f'mean {accs.mean():.1f}, max - min {np.ptp(accs):.1f} points')

validation accuracy (%): 81.0 80.3 81.6 81.9 81.7
mean 81.3, max - min 1.5 points


Even though all five runs execute identical code and hyperparameters, their
final accuracies spread by one to four percentage points around 81%, confirming
that a 0.3-point jump from a single run is pure noise. Sound empirical
comparisons always report the mean and spread across multiple seeds.

To verify that seed choice alone drove those differences, we rerun `seed=0`
and compare both accuracy and every weight against the first run:

In [7]:
acc, w = train_classifier(0, data)
diff = float(tf.reduce_max(tf.abs(w - runs[0][1])))
print('same accuracy:', acc == runs[0][0],
      '| same weights:', bool(tf.reduce_all(w == runs[0][1])),
      f'| largest difference {diff:.2g}')

same accuracy: True | same weights: True | largest difference 0


The rerun matches the first run's validation accuracy identically. Whether every
single weight matches down to the last bit depends on whether the GPU sums
floating-point numbers in a fixed order, something seeds alone do not control
(that section).

On a GPU, TensorFlow's bias-gradient reduction sums terms in non-deterministic thread order, so two identically seeded GPU runs can differ by $\sim 10^{-6}$ in their lowest bits.

## Deterministic Arithmetic

Seeding fixes *which* random numbers a run draws, not *how* floating-point
arithmetic combines them. Because every floating-point addition rounds to the
nearest representable value, floating-point addition is *not* associative:
changing summation order can change the result.

Three float32 numbers make the effect unmistakable. Near $10^8$, adjacent
float32 numbers lie 8 apart (that section), so $-10^8 + 1$ rounds
right back to $-10^8$:

In [8]:
a, b, c = [tf.constant(v, dtype=tf.float32) for v in (1e8, -1e8, 1.0)]
print(float((a + b) + c), float(a + (b + c)))  # same numbers, regrouped

1.0 0.0


Simply shifting parentheses changes the answer from `1.0` to `0.0`. In long
reductions the discrepancy is subtler but pervasive: summing a million floats
forward versus backward routinely differs in the lowest bits.

On a CPU with a fixed thread count, reduction order usually stays constant, so
seeded runs repeat cleanly. On a GPU, thousands of threads race in parallel, and
several fast kernels (including embedding gradients, scatter-adds, certain
convolution backward passes, and reductions) accumulate partial results via
*atomic additions* that land in whatever order threads finish. Two identically
seeded runs on the same GPU can therefore diverge in their lowest bits, which
chaotic gradient steps eventually amplify into visibly distinct loss curves.
Every framework provides a determinism mode (or alternative kernel) that forces
a fixed summation order, typically at some speed cost. Below we observe this directly
on a GPU *scatter-add* summing one million random numbers into ten bins twice in
a row.

`tf.config.experimental.enable_op_determinism()` forces deterministic GPU kernels (such as `tf.math.unsorted_segment_sum`) for the rest of the process:

In [9]:
if tf.config.list_physical_devices('GPU'):
    src = tf.random.normal((2**20,))  # a million numbers
    idx = tf.random.uniform((2**20,), maxval=10, dtype=tf.int32)  # their bins
    for deterministic in (False, True):
        if deterministic:
            tf.config.experimental.enable_op_determinism()
        a, b = [tf.math.unsorted_segment_sum(src, idx, 10) for _ in range(2)]
        print(f'deterministic {deterministic!s:5}: bitwise equal '
              f'{bool(tf.reduce_all(a == b))}, largest difference '
              f'{float(tf.reduce_max(tf.abs(a - b))):.2g}')
else:
    print('no GPU: on a CPU this sum adds in a fixed order')

deterministic False: bitwise equal False, largest difference 0.0057
deterministic True : bitwise equal True, largest difference 0


Without determinism enabled, two consecutive scatter-adds on the exact same GPU
data differ by a few thousandths on sums of a few hundred; with a deterministic
kernel, both calls match bit for bit.

With the switch on, random operations must have a seed, since an operation
that seeds itself from the operating system is nondeterministic too. We can
demonstrate the rule by clearing the global seed:

In [10]:
tf.config.experimental.enable_op_determinism()  # no-op if already on
tf.random.set_seed(None)  # forget the global seed for a moment
try:
    tf.random.normal((2,))
except RuntimeError as e:
    print(str(e).split('.')[0])
# determinism stays on and needs a seed, though not a particular one
tf.keras.utils.set_random_seed(random.SystemRandom().getrandbits(31))

Random ops require a seed to be set when determinism is enabled


Remember the proper scope of bitwise reproducibility: it holds only on a single
machine, software stack, and hardware configuration; no framework promises
identical bits across library upgrades, different GPU architectures, or CPU vs.
GPU. Bitwise determinism is an invaluable *debugging* tool for bisecting the
exact step where two runs diverge. Scientific reproducibility is both looser and
stronger: any genuine improvement ("this architectural change improves
accuracy") must hold robustly across multiple seeds, reported as mean and spread
just as in that section (~81%, spanning 1–4 points). As with
16-bit formats (that section), any claim that flips under a new seed
or last-bit rounding needs stronger evidence.

## Model Inspection with Hooks

Repeating a bug reliably proves it is real, not *where* inside the network it
originates. Inserting print statements into every layer's `forward` method is
clumsy, and impractical for third-party library models whose source code you do
not want to fork. A *hook* solves this cleanly: it is a callback registered from
outside the model that the framework invokes automatically before or after a
layer executes.

Keras has no external hook API; instead, we either build a secondary `tf.keras.Model` exposing intermediate symbolic tensors of a functional graph, or log tensors inside a custom `call`.

![The `__call__` wrapper as a pipeline: input flows through pre-hooks, then forward, then hooks, to the output, with the two hook stages dashed and orange against forward's solid blue, and a side arrow from the hooks stage to an observer that can capture, check, or modify.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-hooks.svg)

For our inspection examples, we build an eight-block residual stack of width 64
between a 20-input layer and a 10-output head (that section),
each block computing $\mathbf{h} = \mathbf{x} + \textrm{body}(\mathbf{x})$ with
Xavier-initialized weights (that section) and zero biases. We
deliberately leave this network unseeded so each run draws fresh random weights,
which is why the diagnostic measurements below are cited as typical ranges.

In [11]:
def dense(num_outputs, **kwargs):  # Xavier weights, zero biases
    return tf.keras.layers.Dense(
        num_outputs, kernel_initializer='glorot_uniform', **kwargs)

class ResidualBlock(tf.keras.layers.Layer):
    def __init__(self, num_hiddens, **kwargs):
        super().__init__(**kwargs)
        self.body = tf.keras.Sequential([
            dense(num_hiddens, activation='relu'), dense(num_hiddens)])

    def call(self, X):
        return X + self.body(X)

inputs = tf.keras.Input(shape=(20,))
taps = [dense(64)(inputs)]  # keep every intermediate tensor
for _ in range(8):
    taps.append(ResidualBlock(64)(taps[-1]))
outputs = dense(10)(taps[-1])
net = tf.keras.Model(inputs, outputs)

### Activation Statistics

In that section, we manually looped through blocks to measure
activation standard deviations across depth. With hooks, we can collect the
exact same profile from an unmodified model in just a few lines on a minibatch
of 256 standard normal inputs:

In [12]:
probe = tf.keras.Model(inputs, taps + [outputs])  # same layers, same weights
X = tf.random.normal((256, 20))
names = ['Dense'] + ['ResidualBlock'] * 8 + ['Dense']
for name, out in zip(names, probe(X)):
    print(f'{name:15s} std {float(tf.math.reduce_std(out)):.2f}')

Dense           std 0.69
ResidualBlock   std 0.83
ResidualBlock   std 0.99
ResidualBlock   std 1.23
ResidualBlock   std 1.44
ResidualBlock   std 1.74
ResidualBlock   std 2.06
ResidualBlock   std 2.50
ResidualBlock   std 3.17
Dense           std 4.65


Activation standard deviations climb steadily with depth, from roughly 0.7 right
after the input layer to around 3–4.5 after the eighth block, matching theory
closely. Xavier initialization gives the $20 \to 64$ input layer weight variance
$2/84$, yielding output variance $20 \cdot 2/84 \approx 0.48$ (standard
deviation 0.69). Each residual block then adds an approximately uncorrelated
body output whose variance is half its input variance (the first linear layer
preserves variance, ReLU halves the mean square, and the second linear layer
preserves it), multiplying total variance by $1.5$ (and standard deviation by
$\sqrt{1.5} \approx 1.22$) at every block. Across eight blocks,
$0.69 \times 1.5^4 \approx 3.5$, right in the middle of our measurements. This
is the exact variance-growth analysis of that section (with factor
$1.5$ here instead of $2$ under He initialization), underscoring why deep
residual branches benefit from scaled-down output projections.

`probe(X)` attaches no persistent state to the model, so nothing needs to be detached or removed afterwards.

### A NaN Finder

When a training loss suddenly turns NaN at step 40,000, your first diagnostic
question is *which layer emitted the first non-finite value?* Attaching a
finiteness hook to every leaf layer pinpoints the culprit in a single forward
pass, because hooks execute in forward execution order. To test our detector, we
deliberately inject a NaN into one weight of the third residual block (index 3):

In [13]:
class Checked(tf.keras.Model):
    def __init__(self, layers):
        super().__init__()
        self.seq, self.first_bad = layers, None  # a plain list is tracked

    def call(self, X):
        for layer in self.seq:
            X = layer(X)
            if self.first_bad is None and not bool(
                    tf.reduce_all(tf.math.is_finite(X))):
                self.first_bad = layer.name
        return X

checked = Checked([dense(64)]
                  + [ResidualBlock(64, name=f'block{k}') for k in range(1, 9)]
                  + [dense(10)])
_ = checked(tf.random.normal((2, 20)))  # build the weights
checked.seq[3].body.layers[0].kernel[0, 0].assign(float('nan'))  # break one
_ = checked(tf.random.normal((2, 20)))
print('first non-finite output in', checked.first_bad)

first non-finite output in block3


The hook immediately identifies the corrupted layer in block 3 without adding a
single print statement inside the layer classes.
Of course, catching a planted NaN is the clean demo case; in a real training
crash, the finder points you to the first *symptom*, while the underlying cause
(such as runaway activation growth culminating in overflow) may begin several
layers earlier.

### Gradient Inspection

We can monitor the backward pass with the same precision. Since forward
activations grew across our eight residual blocks, do gradients grow in reverse?
Using the mean squared output on 256 standard normal inputs as our loss, we log
the gradient norm at each residual block's output alongside the weight and bias
gradient norms of the first linear layer in block 3.

`tape.gradient(loss, [acts[1:], net.trainable_variables])` computes gradients with respect to both intermediate functional outputs and trainable variables in one call:

In [14]:
with tf.GradientTape() as tape:
    *acts, out = probe(X)  # every layer's output, then the stack's
    loss = tf.reduce_mean(out ** 2)
block_grads, grads = tape.gradient(
    loss, [acts[1:], net.trainable_variables])
print('block output gradients:',
      ' '.join(f'{float(tf.norm(g)):.3g}' for g in block_grads))
kernel, bias = grads[10:12]  # block 3, first layer
print(f'kernel {float(tf.norm(kernel)):.3g}, bias {float(tf.norm(bias)):.3g}')

block output gradients: 1.54 1.28 0.906 0.709 0.543 0.428 0.345 0.26
kernel 25.8, bias 6.46


Reading backward from the eighth block to the first, the output gradient norm
grows by roughly $1.2\times$ to $1.4\times$ per block (five- to sixfold across
the stack), mirroring the forward $1.22\times$ growth for the exact same reason:
on the backward pass, each residual block adds the branch gradient back onto the
skip-connection gradient. Isolated parameter gradient norms mean little on their
own, but comparing them across depth or tracking them across training steps
reveals vanishing or exploding dynamics immediately (see the exercises).

Remember what these debugging tools do and do not solve: they cannot force two
different machines to match bit for bit, nor can a single run settle whether an
experimental gain is real (for that, compare distributions across multiple seeds).
Yet combined with resumable checkpoints (that section), you now have
every piece needed to replay a failing job right up to the crash and attach a
diagnostic hook to isolate the exact layer at fault.

The 8-billion-parameter language model that opened this chapter has not shrunk
(it still weighs 32 GB in fp32), but nothing about how frameworks handle it is a black
box anymore. Constructing architectures from configs, auditing parameters and
buffers, initializing deep stacks, building custom layers and gradients, training
in mixed precision, saving crash-safe checkpoints, managing GPU memory, and
inspecting live runs all operate over the exact same tree of modules. Equipped
with this engineering toolkit, we are ready to move on to architectures built
around the structure of real-world data, starting with convolutional neural
networks in that section.

## Summary

Every random choice in training (initialization, shuffling, dropout, and data
augmentation) traces back to pseudorandom number generators controlled by seeds,
with private generator objects or explicit keys preventing unrelated code from
perturbing a stream. Because floating-point addition is non-associative and
parallel GPU atomic operations sum in arbitrary thread order, bit-for-bit
repeatability also requires deterministic kernels and holds only on one machine
and software stack, making bitwise determinism a debugging tool while scientific
conclusions must hold across multiple seeds. Finally, forward and backward
hooks (and their framework equivalents) let us inspect activation scales,
gradient norms, and first-occurrence NaNs inside unmodified models.

## Exercises

1. [code] **Residual variance budget.** Override `call` in
   `ResidualBlock` so that it stashes the
   variance of its input, of its body's output, and of its own output, and
   run a freshly built `Checked` model on one batch. Let $\mathbf{x}_k$,
   $\mathbf{b}_k$ and $\mathbf{h}_k$ denote these three tensors for block
   $k$, with each variance taken over all entries. The standard deviations
   in the table of that section grow because each
   block adds its body's output to the stream. Measure how closely

   $$\textrm{Var}[\mathbf{h}_k] = \textrm{Var}[\mathbf{x}_k] + \textrm{Var}[\mathbf{b}_k]$$

   holds for every block, and name the term this equation omits.
2. [code] **Seeds and data loading.** ● Work in a fresh process, since the
   section has already called `enable_op_determinism`, which cannot be
   undone. Build a `tf.data` pipeline over `Dataset.range(64)` that
   shuffles with `shuffle(64, seed=0)` and adds noise to each element in a
   `map` with `num_parallel_calls=4`, and start every run with
   `tf.keras.utils.set_random_seed(0)`.
    1. Draw the noise from NumPy's global generator through
       `tf.numpy_function`. Predict whether two runs attach the same noise
       to the same element under the default `deterministic=True`, and
       justify the prediction from where the randomness is drawn. Verify
       over five runs.
    1. Repeat the five runs twice more: once with
       `tf.config.experimental.enable_op_determinism()` called at program
       start, and once without it but with the noise drawn by
       `tf.random.stateless_normal`, seeded from the element's value.
       Explain each outcome.
3. [code] **Divergence monitor.** ● Turn the finiteness check of
   that section into a per-step training monitor. Rebuild
   the `Checked` model and train it with SGD at learning rate 1 on the
   mean squared error between its output and standard normal targets,
   using one fixed batch of 256 standard normal inputs. Report the first
   step at which a layer's output is non-finite, and the first such layer
   by depth. Check the loss at every step as well, and report whether it
   becomes non-finite before any layer output does.
4. [code] **Gradient norms.** ● Rebuild `net`, multiply the kernel of
   every `Dense` layer inside its residual blocks by 3, and use the mean
   squared output on a fixed batch as the loss.
    1. Compute the norm of each block's output gradient as in this
       section, and each block's parameter-gradient norm from the
       gradients of the trainable variables. Plot both families of norms
       against depth.
    1. Express the gradient of a linear layer's kernel through the layer's
       input and its output gradient. Use this expression to explain why
       the two depth profiles differ, and state which family to monitor for
       an exploding backward signal.

[Discussions](https://d2l.discourse.group/t/36759)